# Dimension Customers Model
**Sources:** crm_customers, erp_customers, erp_customer_location  
**Target:** dim_customers

**Columns to include:**  
--- Columns in crm_customers ---  
customer_id = True  
firstname = True  
lastname = True  
marital_status = True  
gender = True  
date_created = True  
category_id = False  
silver_updated_at = False  

--- Columns in erp_customers ---  
birth_date = True  
gender = True  
customer_id = True  
category_id = False  
silver_updated_at = False   

--- Columns in erp_customer_location ---  
country = True  
customer_id = True  
category_id = False  
silver_updated_at = False  

In [0]:
%sql
USE CATALOG databricks_bootcamp_dwb;
USE SCHEMA silver;

In [0]:
tables = ["crm_customers", "erp_customers", "erp_customer_location"]

for t in tables:
    print(f"--- {t} ---")
    display(spark.table(t).limit(5))
    print()

max_rows = 0
for t in tables:
    count = spark.table(t).count()
    if max_rows < count:
        max_rows = count
    print(f"{t}: {count:,} rows")

print(f"\nMaximum rows: {max_rows:,}")

# Perform Business Transformations and Modeling

In [0]:
# Join the three source tables and select required columns
df = spark.sql("""
    SELECT 
        c.customer_id,
        c.firstname,
        c.lastname,
        c.marital_status,
        c.gender,
        c.date_created,
        e.birth_date,
        l.country
    FROM crm_customers c
    LEFT JOIN erp_customers e ON c.customer_id = e.customer_id
    LEFT JOIN erp_customer_location l ON c.customer_id = l.customer_id
""")

# Write Gold Table

In [0]:
# Write to gold layer
df.write.format("delta").mode("overwrite").saveAsTable("databricks_bootcamp_dwb.gold.dim_customers")

# Sanity Checks
- Check for duplicates
- Ensure northstart metrics from bronze tables are preserved

In [0]:
%sql
SELECT *
FROM databricks_bootcamp_dwb.gold.dim_customers
LIMIT 100;

In [0]:
# Load dim_customers and count rows
df = spark.table("databricks_bootcamp_dwb.gold.dim_customers")
df_rows = df.count()

# Print comparison
print(f"dim_customers: {df_rows:,} rows")
print(f"Maximum rows from source tables: {max_rows:,}")

if df_rows == max_rows:
    print("✓ Row counts match! No unforseen duplication.")
else:
    print(f"⚠ Row count mismatch: Check joins for potential duplication")